# Part 3 : Snowflake CoWork

## このノートブックでやること
1. **データ確認** — バナー広告クリエイティブと配信実績データを確認する
2. **バナー画像の AI 分析** — 画像から5つのクリエイティブ特徴を自動抽出する
3. **Semantic View の作成** — データの意味と関係を定義する
4. **Snowflake CoWork で質問してみよう** — 成功例を体験する
5. **CoWork が答えられない質問をしてみよう** — 失敗例から原因を理解する
6. **Semantic View を改善する** — 効果スコアのメトリクスを追加する
7. **再チャレンジ！** — 改善後に同じ質問をして違いを確認する

## シナリオ
金融系 Web バナー広告の **A/B テストデータ** を使って、「どんなクリエイティブが高い効果スコアを上げるか？」を分析します。

| テーブル | 内容 |
|---|---|
| `raw_ad_creatives` | 広告配信実績（インプレッション・クリック・コンバージョンなど） |
| `gold_ad_image_analysis` | AI が画像から抽出したクリエイティブ特徴量（5項目） |

In [ ]:
-- 環境設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE GLACIERSTYLE_WH;
USE DATABASE GLACIERSTYLE_DB;
USE SCHEMA EC_ANALYTICS_SCHEMA;


## 1. データ確認

バナー広告データ（`raw_ad_creatives`）を確認します。

このテーブルには以下の情報が含まれています：
- **クリエイティブ情報**: 広告名、商品ジャンル（投資 / 保険 / クレジットカード）
- **配信実績**: インプレッション数・クリック数・コンバージョン数・広告費用
- **画像参照**: バナー画像ファイル名（`image_file_path`）

まず、金融バナー広告データをテーブルに投入してから内容を確認します。

今回 SI で分析に使うデータを確認しておきましょう。

### 使用するテーブル

| テーブル | 内容 |
|---|---|
| `raw_sns_mentions` | インフルエンサーの SNS 投稿（いいね・RT 数など） |
| `fact_orders` | 注文データ（売上金額・日時など） |
| `dim_products` | 商品マスタ（商品名・カテゴリなど） |

In [ ]:
-- ========================================================
-- 金融バナー広告データを raw_ad_creatives に投入
-- （再実行時は既存の FIN- データを削除して再挿入）
-- ========================================================
DELETE FROM raw_ad_creatives WHERE creative_id LIKE 'FIN-%';

INSERT INTO raw_ad_creatives
    (creative_id, campaign_id, creative_name, creative_type, image_file_path,
     target_segment, platform, start_date, end_date,
     impressions, clicks, conversions, spend)
VALUES
  ('FIN-001','CAM-FIN-01','投資信託_マスコット_シンプル',   'image','ad_001.png','投資',           'web','2025-01-01','2025-01-31', 45000,  540, 27,135000.00),
  ('FIN-002','CAM-FIN-01','NISA_グラフ複雑_数字強調',      'image','ad_002.png','投資',           'web','2025-01-01','2025-01-31', 38000,  684, 48,114000.00),
  ('FIN-003','CAM-FIN-01','生命保険_家族写真_シンプル',     'image','ad_003.png','保険',           'web','2025-01-01','2025-01-31', 62000, 1550, 62,155000.00),
  ('FIN-004','CAM-FIN-01','がん保険_マスコット_数字強調',   'image','ad_004.png','保険',           'web','2025-01-01','2025-01-31', 51000,  765, 46,127500.00),
  ('FIN-005','CAM-FIN-01','クレカ_カード実物_数字強調',     'image','ad_005.png','クレジットカード','web','2025-01-01','2025-01-31', 70000, 1330, 67,196000.00),
  ('FIN-006','CAM-FIN-01','クレカ_マスコット_数字強調',     'image','ad_006.png','クレジットカード','web','2025-01-01','2025-01-31', 58000,  928, 46,162400.00),
  ('FIN-007','CAM-FIN-01','医療保険_女性写真_シンプル',     'image','ad_007.png','保険',           'web','2025-01-01','2025-01-31', 55000, 1265, 38,137500.00),
  ('FIN-008','CAM-FIN-01','iDeCo_グラフ複雑_数字強調',     'image','ad_008.png','投資',           'web','2025-01-01','2025-01-31', 42000,  714, 57,126000.00);


In [ ]:
-- 投入したデータを確認
SELECT
    creative_id,
    creative_name,
    image_file_path,
    target_segment,
    impressions,
    clicks,
    conversions,
    ROUND(spend)                                            AS spend,
    ROUND(DIV0(clicks, impressions) * 100, 2)             AS ctr_pct,
    ROUND(DIV0(conversions, clicks) * 100, 2)             AS cvr_pct
FROM raw_ad_creatives
WHERE creative_id LIKE 'FIN-%'
ORDER BY creative_id;


## 2. バナー広告画像の AI 分析

`AI_COMPLETE` を使って、バナー画像から5つのクリエイティブ特徴量を自動抽出します。

| 特徴量 | 型 | 説明 |
|---|---|---|
| `has_person` | BOOLEAN | 人物（顔・体・手など）が写っているか |
| `has_mascot` | BOOLEAN | マスコットキャラクターが登場するか |
| `color_tone` | VARCHAR | 色調（暖色系 / 寒色系） |
| `is_complex` | BOOLEAN | 情報量が多い複雑な構成か |
| `has_number_highlight` | BOOLEAN | 数字（%・金利・円）が強調されているか |

抽出した特徴量は `gold_ad_image_analysis` テーブルに保存します。

SNS の投稿には**画像**が添付されています。テキストデータだけでなく、画像の特徴もバズスコアに影響するかもしれません。

Snowflake の AI 関数を使って、投稿画像から特徴を自動抽出します。

### 抽出する特徴

| 特徴 | 内容 |
|---|---|
| 人物の有無 | 人が写っているか |
| 表情 | ポジティブ / ネガティブ / ニュートラル |
| 画像の明るさ | 明るい / 暗い |
| 商品の視認性 | 商品がはっきり映っているか |

In [ ]:
-- ステージ上のバナー画像ファイルを確認（8件あること）
SELECT
    relative_path,
    size,
    last_modified
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'images/part3/ad_%.png'
ORDER BY relative_path;


In [ ]:
-- 1枚の画像で AI 分析のテスト（ad_001.png）
-- JSON が正しく返ってくるか確認する
SELECT
    relative_path,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-4-6',
                PROMPT(
                    $$以下の金融広告バナー画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

has_person: true or false
  -- 広告バナーに人物（顔・体・手など身体の一部）が写っているか
has_mascot: true or false
  -- 広告バナーにマスコットキャラクター（動物・キャラクター・ゆるキャラなど）が登場するか。実在の人物は含まない
color_tone: "暖色系" or "寒色系"
  -- 画像全体の色調。赤・オレンジ・黄が主体なら暖色系、青・紺・緑が主体なら寒色系
is_complex: true or false
  -- グラフ・表・複数のテキストブロックなど情報要素が多く複雑な構成か。テキストと要素が少なくシンプルならfalse
has_number_highlight: true or false
  -- 利率（%）・金額（円）・期間・還元率など具体的な数字が大きく・目立つ形で強調されているか

画像: {0}$$,
                    TO_FILE('@DATA_STAGE', relative_path)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    ) AS features_json
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path = 'images/part3/ad_001.png';


In [ ]:
-- 全バナー画像を分析して gold テーブルに保存
-- ※ AI_COMPLETE が全件実行されるので数秒〜数十秒かかります
CREATE OR REPLACE TABLE gold_ad_image_analysis AS
SELECT
    REGEXP_SUBSTR(relative_path, 'ad_[0-9]+[.]png$')    AS image_file,
    relative_path,
    PARSE_JSON(
        REGEXP_REPLACE(
            AI_COMPLETE(
                'claude-sonnet-4-6',
                PROMPT(
                    $$以下の金融広告バナー画像を分析して、JSON形式で結果を返してください。
JSONオブジェクトのみ返してください。説明文やコードブロック記法は不要です。

has_person: true or false
  -- 広告バナーに人物（顔・体・手など身体の一部）が写っているか
has_mascot: true or false
  -- 広告バナーにマスコットキャラクター（動物・キャラクター・ゆるキャラなど）が登場するか。実在の人物は含まない
color_tone: "暖色系" or "寒色系"
  -- 画像全体の色調。赤・オレンジ・黄が主体なら暖色系、青・紺・緑が主体なら寒色系
is_complex: true or false
  -- グラフ・表・複数のテキストブロックなど情報要素が多く複雑な構成か。テキストと要素が少なくシンプルならfalse
has_number_highlight: true or false
  -- 利率（%）・金額（円）・期間・還元率など具体的な数字が大きく・目立つ形で強調されているか

画像: {0}$$,
                    TO_FILE('@DATA_STAGE', relative_path)
                )
            ),
            '^[^{]*|[^}]*$', ''
        )
    )                                                     AS features_json,
    features_json:has_person::BOOLEAN                    AS has_person,
    features_json:has_mascot::BOOLEAN                    AS has_mascot,
    features_json:color_tone::VARCHAR                    AS color_tone,
    features_json:is_complex::BOOLEAN                    AS is_complex,
    features_json:has_number_highlight::BOOLEAN          AS has_number_highlight
FROM DIRECTORY(@DATA_STAGE)
WHERE relative_path LIKE 'images/part3/ad_%.png'
ORDER BY relative_path;

-- 結果確認
SELECT * FROM gold_ad_image_analysis ORDER BY image_file;


## 3. Semantic View を作成する

**Semantic View（セマンティックビュー）** は、CoWork が「このデータは何を意味するか」を理解するためのメタデータ定義です。

- テーブルの意味や関係性を定義する
- ビジネス指標（メトリクス）をあらかじめ定義しておく
- CoWork はこの定義をもとに自然言語クエリを SQL に変換する

### ポイント

Semantic View に **定義されていないメトリクスや結合は CoWork が答えられません。**
後のセクションでこれを体験します。

### 今回作成する Semantic View（Before 版）

| 要素 | 内容 |
|---|---|
| ベーステーブル | `raw_ad_creatives`（広告配信実績） |
| メトリクス | インプレッション・クリック・コンバージョン・CTR |
| ※ 未定義 | 効果スコア、画像分析テーブルとの結合 |

In [ ]:
-- =====================================================
-- Semantic View の作成（Before 版）
-- raw_ad_creatives のみ。効果スコアと画像分析は未定義。
-- =====================================================
CREATE OR REPLACE SEMANTIC VIEW sv_ad_creative_analysis
  TABLES (
    raw_ad_creatives PRIMARY (
      DIMENSIONS (
        creative_id     COMMENT '広告クリエイティブID',
        creative_name   COMMENT '広告名',
        image_file_path COMMENT 'バナー画像ファイル名',
        target_segment  COMMENT '商品ジャンル（投資 / 保険 / クレジットカード）',
        platform        COMMENT '配信プラットフォーム',
        start_date      COMMENT '配信開始日',
        end_date        COMMENT '配信終了日'
      )
      METRICS (
        total_impressions AS SUM(impressions)                      COMMENT '合計インプレッション数',
        total_clicks      AS SUM(clicks)                          COMMENT '合計クリック数',
        total_conversions AS SUM(conversions)                     COMMENT '合計コンバージョン数',
        total_spend       AS SUM(spend)                           COMMENT '合計広告費用（円）',
        ctr               AS DIV0(SUM(clicks), SUM(impressions))  COMMENT 'クリック率 = clicks ÷ impressions'
      )
    )
  );


In [ ]:
-- 作成した Semantic View を確認
DESCRIBE SEMANTIC VIEW sv_ad_creative_analysis;


## 4. Snowflake CoWork で質問してみよう（成功例）

Semantic View の準備ができました。いよいよ CoWork を使ってみましょう。

### CoWork の起動方法

1. Snowsight の左メニューから **「Snowflake Intelligence」** をクリック
2. 右上の **「Add data」** から `sv_ad_creative_analysis` を選択
3. チャット欄に質問を入力

### 試してみる質問（シンプルな集計）

まずは CoWork が確実に答えられる**シンプルな質問**を試してみましょう。

```
クリック率（CTR）が高い商品ジャンルを教えてください
```

```
合計広告費用が最も多いジャンルはどこですか？
```

```
コンバージョン数の合計を商品ジャンル別に見せてください
```

> **確認ポイント:** CoWork が SQL を自動生成して結果を返してくれることを確認しましょう。

In [ ]:
-- 参考: CoWork が生成する SQL と同等のクエリ（手動で確認）
-- CoWork の回答と結果が一致するか検証に使えます
SELECT
    target_segment,
    SUM(impressions)                                           AS total_impressions,
    SUM(clicks)                                               AS total_clicks,
    SUM(conversions)                                          AS total_conversions,
    ROUND(DIV0(SUM(clicks), SUM(impressions)) * 100, 2)      AS ctr_pct
FROM raw_ad_creatives
WHERE creative_id LIKE 'FIN-%'
GROUP BY target_segment
ORDER BY ctr_pct DESC;


## 5. CoWork が答えられない質問をしてみよう（失敗例）

今度は少し**踏み込んだ質問**を CoWork にしてみましょう。

### 試してみる質問

```
効果スコアが高い広告クリエイティブの傾向を教えてください
```

```
人物が写っている広告とそうでない広告で、効果スコアに差はありますか？
```

```
暖色系と寒色系のバナーでクリック率に差はありますか？
```

> **予想:** CoWork はこれらの質問に答えられないか、的外れな回答をするはずです。

### なぜ答えられないのか？

**2つの理由があります。**

1. **「効果スコア」という概念が Semantic View に定義されていないから**
   - 効果スコア = CTR × CVR × 10,000（社内 KPI）
   - この計算式が SV に登録されていないため、CoWork は意味を理解できない

2. **画像分析テーブル（`gold_ad_image_analysis`）が SV に含まれていないから**
   - `has_person`・`color_tone` などのクリエイティブ特徴量が「地図」に載っていない
   - `raw_ad_creatives` との結合も定義されていない

→ **Semantic View を改善して、このような質問にも答えられるようにしましょう！**

## 6. Semantic View を改善する

**効果スコア** のメトリクスと **画像分析テーブル** を Semantic View に追加します。

### 効果スコアの定義

| 指標 | 計算式 | 意味 |
|---|---|---|
| CTR | clicks ÷ impressions | クリック率（表示からクリックされる確率） |
| CVR | conversions ÷ clicks | コンバージョン率（クリックから成約する確率） |
| **効果スコア** | **CTR × CVR × 10,000** | 1回表示あたりの成約確率を10,000倍した社内 KPI |

効果スコアが高いほど「1回の広告表示から成約に至る確率が高い」クリエイティブです。

### 改善するポイント

1. `cvr`・`kouka_score`（効果スコア）メトリクスの追加
2. `gold_ad_image_analysis`（画像分析テーブル）を SV に追加
3. `raw_ad_creatives` と `gold_ad_image_analysis` の **RELATIONSHIP** を定義
4. **Verified Query**（検証済みクエリ）の追加

> **Verified Query とは？**
> よく聞かれる質問とその正解 SQL をペアで登録することで、CoWork の回答精度が向上します。

In [ ]:
-- =====================================================
-- Semantic View を改善版に再作成（After 版）
-- 効果スコア + 画像分析テーブル + Verified Query を追加
-- =====================================================
CREATE OR REPLACE SEMANTIC VIEW sv_ad_creative_analysis
  TABLES (
    raw_ad_creatives PRIMARY (
      DIMENSIONS (
        creative_id     COMMENT '広告クリエイティブID',
        creative_name   COMMENT '広告名',
        image_file_path COMMENT 'バナー画像ファイル名（gold_ad_image_analysis との結合キー）',
        target_segment  COMMENT '商品ジャンル（投資 / 保険 / クレジットカード）',
        platform        COMMENT '配信プラットフォーム',
        start_date      COMMENT '配信開始日',
        end_date        COMMENT '配信終了日'
      )
      METRICS (
        total_impressions AS SUM(impressions)                                       COMMENT '合計インプレッション数',
        total_clicks      AS SUM(clicks)                                            COMMENT '合計クリック数',
        total_conversions AS SUM(conversions)                                       COMMENT '合計コンバージョン数',
        total_spend       AS SUM(spend)                                             COMMENT '合計広告費用（円）',
        ctr               AS DIV0(SUM(clicks), SUM(impressions))                   COMMENT 'クリック率 = clicks ÷ impressions',
        cvr               AS DIV0(SUM(conversions), SUM(clicks))                   COMMENT 'コンバージョン率 = conversions ÷ clicks',
        kouka_score       AS DIV0(SUM(conversions), SUM(impressions)) * 10000      COMMENT '効果スコア = CTR × CVR × 10000（社内 KPI）'
      )
    ),
    gold_ad_image_analysis (
      DIMENSIONS (
        image_file           COMMENT 'バナー画像ファイル名（raw_ad_creatives.image_file_path と結合）',
        has_person           COMMENT '人物が写っているか（true / false）',
        has_mascot           COMMENT 'マスコットキャラクターが登場するか（true / false）',
        color_tone           COMMENT '色調（暖色系 / 寒色系）',
        is_complex           COMMENT '情報量が多い複雑なレイアウトか（true / false）',
        has_number_highlight COMMENT '数字（%・金利・円）が強調されているか（true / false）'
      )
    )
  )
  RELATIONSHIPS (
    gold_ad_image_analysis (image_file) REFERENCES raw_ad_creatives (image_file_path)
  )
  VERIFIED QUERIES (
    vq1 QUESTION '効果スコアが高い広告クリエイティブの傾向を教えてください'
        VERIFIED_QUERY 'SELECT i.has_person, i.has_mascot, i.color_tone, i.is_complex, i.has_number_highlight, ROUND(DIV0(SUM(r.conversions), SUM(r.impressions)) * 10000, 2) AS kouka_score FROM raw_ad_creatives r JOIN gold_ad_image_analysis i ON r.image_file_path = i.image_file GROUP BY 1,2,3,4,5 ORDER BY kouka_score DESC',
    vq2 QUESTION '人物が写っている広告とそうでない広告で効果スコアに差はありますか'
        VERIFIED_QUERY 'SELECT i.has_person, ROUND(AVG(DIV0(r.conversions, r.impressions)) * 10000, 2) AS avg_kouka_score FROM raw_ad_creatives r JOIN gold_ad_image_analysis i ON r.image_file_path = i.image_file GROUP BY i.has_person ORDER BY avg_kouka_score DESC'
  );


In [ ]:
-- 効果スコアを手動で確認（CoWork の回答と一致するか検証）
SELECT
    r.target_segment,
    r.creative_name,
    i.has_person,
    i.has_mascot,
    i.color_tone,
    i.is_complex,
    i.has_number_highlight,
    ROUND(DIV0(SUM(r.clicks), SUM(r.impressions)) * 100, 2)        AS ctr_pct,
    ROUND(DIV0(SUM(r.conversions), SUM(r.clicks)) * 100, 2)        AS cvr_pct,
    ROUND(DIV0(SUM(r.conversions), SUM(r.impressions)) * 10000, 2) AS kouka_score
FROM raw_ad_creatives r
JOIN gold_ad_image_analysis i ON r.image_file_path = i.image_file
WHERE r.creative_id LIKE 'FIN-%'
GROUP BY r.target_segment, r.creative_name,
         i.has_person, i.has_mascot, i.color_tone, i.is_complex, i.has_number_highlight
ORDER BY kouka_score DESC;


## 7. 再チャレンジ！（改善後）

Semantic View を改善しました。もう一度 CoWork に同じ質問をしてみましょう。

### 再度試してみる質問

```
効果スコアが高い広告クリエイティブの傾向を教えてください
```

```
人物が写っている広告とそうでない広告で、効果スコアに差はありますか？
```

```
暖色系と寒色系のバナーでクリック率に差はありますか？
```

> **確認ポイント:**
> - 先ほど答えられなかった質問に、今度は正しく回答できるか確認しましょう
> - CoWork が画像特徴量と配信実績を **JOIN した SQL** を自動生成していることを確認しましょう

### まとめ

| | 改善前 | 改善後 |
|---|---|---|
| CTR・ジャンルの質問 | ✅ 正しく回答 | ✅ 正しく回答 |
| 効果スコアの質問 | ❌ 答えられない | ✅ 正しく回答 |
| 画像特徴との相関 | ❌ 答えられない | ✅ 正しく回答 |

**Semantic View の定義がそのまま CoWork の「答えられる範囲」になります。**
ビジネス要件が追加されるたびに Semantic View を育てることで、CoWork はどんどん賢くなります。

## お疲れ様でした！

### Part 3 で学んだこと

- **AI_COMPLETE** を使って画像を分析し、クリエイティブ特徴量を自動抽出した
- **Semantic View** でテーブルの意味・メトリクス・テーブル間の結合を定義した
- **Snowflake Intelligence（CoWork）** で自然言語によるデータ分析を体験した
- SV に **効果スコア** を追加することで、CoWork の回答範囲を広げた

### 実業務への応用

- 広告配信データに AI 画像分析を組み合わせて、クリエイティブ戦略を最適化できる
- 社内 KPI（効果スコアなど）を SV に定義することで、誰でも**同じ定義**で分析が可能
- Verified Query で よく聞かれる質問への回答精度をさらに向上できる

### 次のステップ

- Part 4: Snowflake Marketplace（外部データとの連携）